
# RSNA Knee — Silver Label Construction v1

## Vị trí trong roadmap

**EDA ✅ → Label Audit ✅ → Silver Label Construction (notebook này) → Baseline MRI V1 → Controlled Experiments**

Notebook này **không train MRI model**. Nó dùng kết quả rút ra từ Label Audit v2 để xây bộ silver labels sạch hơn cho 4,349 study không có gold labels.

### Các vấn đề từ Label Audit v2 cần sửa

1. `4-Source Merged / report_labels_v5.csv` đạt AUC=1.0 trên 58 gold và có đúng 696 giá trị binary (= 58×12)  
   → coi là **gold-contaminated reference**, không dùng để đánh giá/weight chính nó.

2. Nhiều file pseudo-label không độc lập:
   - Hybrid + Steven v4 blend gần như cùng family
   - 2 file GPT-5.6-Sol giống nhau
   - Steven full + v2 cùng family
   - Yunus là family riêng

3. Không nên majority-vote theo **file**, vì family có nhiều file sẽ được nhiều phiếu hơn.

4. Giữ **soft labels**. Không ép `>=0.5 → 1` ở giai đoạn tạo supervision.

---

## Output chính

Notebook tạo 3 candidate silver-label sets:

- `L0_hybrid_family.csv` — reference mạnh, đơn giản.
- `L1_family_mean.csv` — **candidate baseline đề xuất**: equal-weight giữa các family độc lập.
- `L2_family_median.csv` — robust consensus.
- `L1_agreement_weights.csv` — weight theo mức độ đồng thuận giữa family, dùng cho experiment loss-weighting sau.
- `L3_gold_guided_EXPERIMENTAL.csv` — chỉ để nghiên cứu, **không dùng làm baseline mặc định** vì có nguy cơ meta-overfit 58 gold.

Ngoài ra xuất:
- source/family manifest
- family-level AUC trên 58 gold
- candidate-level AUC
- các case disagreement cao để audit report/domain knowledge

GPU: **không cần**.


In [ ]:

# ============================================================
# 0. Imports & config
# ============================================================
from pathlib import Path
import os, re, json, math, warnings, zipfile
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

SEED = 42
BOOTSTRAP_ITERS = 500

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

print("Ready.")


In [ ]:

# ============================================================
# 1. Locate competition train.csv and gold studies
# ============================================================
def find_competition_root():
    candidates = []
    for p in Path("/kaggle/input").rglob("train.csv"):
        try:
            df = pd.read_csv(p, nrows=2)
            if {"StudyInstanceUID", "Report", "ACL", "MCL"}.issubset(df.columns):
                candidates.append(p.parent)
        except Exception:
            pass

    if not candidates:
        raise FileNotFoundError(
            "Attach competition input: RSNA Knee Abnormality Detection"
        )

    comp = [x for x in candidates if "/competitions/" in str(x)]
    return comp[0] if comp else candidates[0]

COMP_ROOT = find_competition_root()
train = pd.read_csv(COMP_ROOT / "train.csv")

train["StudyInstanceUID"] = train["StudyInstanceUID"].astype(str)
train["n_available_labels"] = train[LABELS].notna().sum(axis=1)

gold = train.loc[
    train["n_available_labels"].eq(len(LABELS)),
    ["StudyInstanceUID", "Report"] + LABELS
].copy()

weak = train.loc[
    ~train["n_available_labels"].eq(len(LABELS)),
    ["StudyInstanceUID", "Report"]
].copy()

TRAIN_UIDS = set(train["StudyInstanceUID"])
GOLD_UIDS = set(gold["StudyInstanceUID"])

print("Competition root:", COMP_ROOT)
print("Train:", len(train))
print("Gold:", len(gold))
print("Weak studies:", len(weak))


In [ ]:

# ============================================================
# 2. Schema helpers
# ============================================================
def norm_name(x):
    x = str(x).strip().lower().replace("'", "")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

ALIASES = {
    "ACL": ["acl", "acl injury", "acl tear", "anterior cruciate ligament"],
    "MCL": ["mcl", "mcl injury", "mcl tear", "medial collateral ligament"],
    "Medial Meniscus": [
        "medial meniscus", "medial meniscus tear",
        "medial meniscal tear", "medial meniscal", "mm"
    ],
    "Lateral Meniscus": [
        "lateral meniscus", "lateral meniscus tear",
        "lateral meniscal tear", "lateral meniscal", "lm"
    ],
    "Medial OA": [
        "medial oa", "medial osteoarthritis",
        "medial tibiofemoral oa", "medial tibiofemoral osteoarthritis"
    ],
    "Lateral OA": [
        "lateral oa", "lateral osteoarthritis",
        "lateral tibiofemoral oa", "lateral tibiofemoral osteoarthritis"
    ],
    "PF OA": [
        "pf oa", "patellofemoral oa",
        "patellofemoral osteoarthritis", "patello femoral oa"
    ],
    "Effusion": ["effusion", "joint effusion"],
    "Synovitis": ["synovitis"],
    "Baker's": [
        "bakers", "bakers cyst", "baker cyst",
        "baker s cyst", "popliteal cyst"
    ],
    "Contusion": [
        "contusion", "bone contusion",
        "bone bruise", "bone bruising"
    ],
    "Fracture": ["fracture", "bone fracture"],
}

ALIAS_TO_LABEL = {}
for label, vals in ALIASES.items():
    ALIAS_TO_LABEL[norm_name(label)] = label
    for x in vals:
        ALIAS_TO_LABEL[norm_name(x)] = label

UID_ALIASES = {
    "studyinstanceuid", "study instance uid",
    "studyuid", "study uid", "studyid", "study id", "uid"
}

def infer_uid_col(df):
    mp = {norm_name(c): c for c in df.columns}
    for a in UID_ALIASES:
        if norm_name(a) in mp:
            return mp[norm_name(a)]
    return None

def infer_label_cols(df):
    found = {}
    for c in df.columns:
        nc = norm_name(c)
        if nc in ALIAS_TO_LABEL:
            found[ALIAS_TO_LABEL[nc]] = c
            continue

        for label in LABELS:
            t = norm_name(label)
            variants = {
                f"{t} score", f"{t} prob", f"{t} probability",
                f"{t} label", f"{t} pred", f"{t} prediction",
                f"pred {t}", f"prob {t}", f"score {t}"
            }
            if nc in variants:
                found[label] = c
                break

    return found

def safe_read(path, nrows=None):
    path = Path(path)
    try:
        s = path.suffix.lower()
        if s == ".csv":
            return pd.read_csv(path, nrows=nrows)
        if s in {".parquet", ".pq"}:
            df = pd.read_parquet(path)
            return df.head(nrows) if nrows else df
        if s == ".json":
            try:
                df = pd.read_json(path)
            except Exception:
                df = pd.read_json(path, lines=True)
            return df.head(nrows) if nrows else df
    except Exception:
        return None
    return None

def coerce_score(s):
    numeric = pd.to_numeric(s, errors="coerce")
    if numeric.notna().mean() >= 0.70:
        return numeric.astype(float)

    mapping = {
        "positive": 1.0, "pos": 1.0, "present": 1.0,
        "yes": 1.0, "true": 1.0,
        "negative": 0.0, "neg": 0.0, "absent": 0.0,
        "no": 0.0, "false": 0.0,
        "uncertain": 0.5, "possible": 0.5, "indeterminate": 0.5,
        "unknown": np.nan, "unmentioned": np.nan, "not mentioned": np.nan,
    }
    return s.astype(str).str.strip().str.lower().map(mapping).astype(float)


In [ ]:

# ============================================================
# 3. Discover direct pseudo-label datasets
# ============================================================
paths = []
dataset_root = Path("/kaggle/input/datasets")

if dataset_root.exists():
    for ext in ["*.csv", "*.parquet", "*.pq", "*.json"]:
        paths.extend(dataset_root.rglob(ext))
else:
    for ext in ["*.csv", "*.parquet", "*.pq", "*.json"]:
        for p in Path("/kaggle/input").rglob(ext):
            sp = str(p)
            if "/notebooks/" in sp or "/competitions/" in sp:
                continue
            paths.append(p)

rows = []

for p in sorted(set(paths), key=str):
    if "submission" in p.name.lower():
        continue

    preview = safe_read(p, nrows=20)
    if preview is None or len(preview) == 0:
        continue

    uid_col = infer_uid_col(preview)
    label_cols = infer_label_cols(preview)

    if uid_col is None or len(label_cols) == 0:
        continue

    full = safe_read(p)
    if full is None:
        continue

    uid_col = infer_uid_col(full)
    label_cols = infer_label_cols(full)

    uid_vals = set(full[uid_col].astype(str))
    train_overlap = len(uid_vals & TRAIN_UIDS)
    gold_overlap = len(uid_vals & GOLD_UIDS)

    if train_overlap == 0:
        continue

    rows.append({
        "path": str(p),
        "dataset_folder": p.parent.name,
        "file_name": p.name,
        "n_rows": len(full),
        "uid_col": uid_col,
        "n_labels": len(label_cols),
        "train_overlap": train_overlap,
        "gold_overlap": gold_overlap
    })

discovery = pd.DataFrame(rows)

if len(discovery) == 0:
    raise RuntimeError(
        "No direct pseudo-label TRAIN dataset found. "
        "Attach the same direct DATASETS used for Label Audit v2."
    )

discovery = discovery.sort_values(
    ["train_overlap", "n_labels", "n_rows"],
    ascending=False
).reset_index(drop=True)

display(discovery)


In [ ]:

# ============================================================
# 4. Normalize sources
# ============================================================
sources = {}
manifest_rows = []

for _, row in discovery.iterrows():
    raw = safe_read(row["path"])
    uid_col = infer_uid_col(raw)
    label_cols = infer_label_cols(raw)

    out = pd.DataFrame({
        "StudyInstanceUID": raw[uid_col].astype(str)
    })

    for label, c in label_cols.items():
        out[label] = coerce_score(raw[c])

    out = out.groupby("StudyInstanceUID", as_index=False).mean(numeric_only=True)

    sid = f"S{len(sources)+1:02d}"
    sources[sid] = out

    vals = out[[c for c in LABELS if c in out.columns]].stack().dropna()

    manifest_rows.append({
        "source_id": sid,
        "dataset_folder": row["dataset_folder"],
        "file_name": row["file_name"],
        "path": row["path"],
        "n_rows": len(out),
        "n_labels": len([c for c in LABELS if c in out.columns]),
        "train_overlap": row["train_overlap"],
        "gold_overlap": row["gold_overlap"],
        "binary_fraction": float(vals.isin([0,1]).mean()) if len(vals) else np.nan
    })

source_manifest = pd.DataFrame(manifest_rows)
display(source_manifest)


In [ ]:

# ============================================================
# 5. Detect exact/near-exact GOLD contamination
# ============================================================
leak_rows = []

for sid, sdf in sources.items():
    m = gold[["StudyInstanceUID"] + LABELS].merge(
        sdf, on="StudyInstanceUID", how="inner", suffixes=("_gold", "_src")
    )

    exact_cells = 0
    total_cells = 0
    per_label_exact = []

    for label in LABELS:
        gcol = label + "_gold"
        scol = label + "_src"
        if scol not in m.columns:
            continue

        g = pd.to_numeric(m[gcol], errors="coerce").to_numpy(float)
        s = pd.to_numeric(m[scol], errors="coerce").to_numpy(float)
        valid = np.isfinite(g) & np.isfinite(s)

        if valid.any():
            eq = np.isclose(g[valid], s[valid], atol=1e-12)
            exact_cells += int(eq.sum())
            total_cells += int(valid.sum())
            per_label_exact.append(float(eq.mean()))

    exact_rate = exact_cells / total_cells if total_cells else np.nan

    # Conservative flag: essentially every gold cell copied exactly.
    gold_contaminated = bool(
        total_cells >= len(gold) * 10 and
        np.isfinite(exact_rate) and
        exact_rate > 0.995
    )

    leak_rows.append({
        "source_id": sid,
        "gold_cells_compared": total_cells,
        "exact_gold_cells": exact_cells,
        "exact_gold_match_rate": exact_rate,
        "min_per_label_exact_rate": min(per_label_exact) if per_label_exact else np.nan,
        "gold_contaminated_reference": gold_contaminated
    })

leak_audit = pd.DataFrame(leak_rows)
source_manifest = source_manifest.merge(leak_audit, on="source_id", how="left")

print("Gold contamination audit:")
display(source_manifest.sort_values("exact_gold_match_rate", ascending=False))

LEAKY_SOURCES = set(
    source_manifest.loc[
        source_manifest["gold_contaminated_reference"],
        "source_id"
    ]
)

print("Excluded from silver construction/evaluation:", sorted(LEAKY_SOURCES))


In [ ]:

# ============================================================
# 6. Pairwise dependency audit
# ============================================================
pair_rows = []
sids = list(sources)

for i in range(len(sids)):
    for j in range(i+1, len(sids)):
        a, b = sids[i], sids[j]
        m = sources[a].merge(
            sources[b], on="StudyInstanceUID", how="inner",
            suffixes=("_a", "_b")
        )

        all_abs = []
        all_hard = []

        for label in LABELS:
            ca, cb = label+"_a", label+"_b"
            if ca not in m.columns or cb not in m.columns:
                continue
            x = pd.to_numeric(m[ca], errors="coerce").to_numpy(float)
            y = pd.to_numeric(m[cb], errors="coerce").to_numpy(float)
            valid = np.isfinite(x) & np.isfinite(y)
            x, y = x[valid], y[valid]
            if len(x):
                all_abs.extend(np.abs(x-y).tolist())
                all_hard.extend(((x>=.5)!=(y>=.5)).astype(float).tolist())

        pair_rows.append({
            "source_a": a,
            "source_b": b,
            "mean_abs_difference_all_labels": float(np.mean(all_abs)) if all_abs else np.nan,
            "hard_disagreement_all_labels": float(np.mean(all_hard)) if all_hard else np.nan,
        })

pairwise_dependency = pd.DataFrame(pair_rows)
display(pairwise_dependency.sort_values("mean_abs_difference_all_labels"))



## 7. Source → Family

Family grouping được dùng để **mỗi pipeline label chỉ có một phiếu**, thay vì mỗi file có một phiếu.

Rules dựa trên tên file/dataset từ Label Audit v2:

- `HYBRID_BLEND`: Hybrid + Steven v4 blend
- `GPT56SOL`: các file GPT-5.6-Sol
- `STEVEN_CORE`: Steven full + v2
- `YUNUS`: Yunus
- source bị phát hiện gold-contaminated → `REFERENCE_LEAKY`, không dùng tạo silver labels

Nếu xuất hiện file lạ, notebook cho vào family riêng `OTHER_<source_id>` để không silently merge sai.


In [ ]:

# ============================================================
# 7. Assign source families
# ============================================================
def assign_family(row):
    sid = row["source_id"]
    if sid in LEAKY_SOURCES:
        return "REFERENCE_LEAKY"

    s = (str(row["dataset_folder"]) + " " + str(row["file_name"])).lower()

    if "gpt56sol" in s or "sol56" in s or "gpt-5.6" in s:
        return "GPT56SOL"

    if "v4hybrid" in s or "hybrid" in s or "v4_blend" in s:
        return "HYBRID_BLEND"

    if "yunus" in s:
        return "YUNUS"

    if "llm_labels_full" in s or "llm_labels_v2" in s:
        return "STEVEN_CORE"

    if "4-source" in s or "4_source" in s or "report_labels_v5" in s:
        return "REFERENCE_LEAKY"

    return f"OTHER_{sid}"

source_manifest["family"] = source_manifest.apply(assign_family, axis=1)

print("Source family manifest:")
display(source_manifest[[
    "source_id","dataset_folder","file_name","family",
    "gold_contaminated_reference","binary_fraction"
]])

print("\nFamily counts:")
display(source_manifest["family"].value_counts().to_frame("n_sources"))


In [ ]:

# ============================================================
# 8. Build one prediction table per independent family
# ============================================================
NONLEAK = source_manifest[
    source_manifest["family"] != "REFERENCE_LEAKY"
].copy()

FAMILIES = sorted(NONLEAK["family"].unique())
print("Independent families:", FAMILIES)

all_uids = train[["StudyInstanceUID"]].copy()
family_tables = {}

for family in FAMILIES:
    members = NONLEAK.loc[
        NONLEAK["family"].eq(family),
        "source_id"
    ].tolist()

    ft = all_uids.copy()

    # Collect all member-source predictions first
    member_cube = all_uids.copy()

    for sid in members:
        sdf = sources[sid]
        rename = {
            label: f"{sid}__{label}"
            for label in LABELS if label in sdf.columns
        }
        member_cube = member_cube.merge(
            sdf.rename(columns=rename),
            on="StudyInstanceUID",
            how="left"
        )

    for label in LABELS:
        cols = [
            f"{sid}__{label}"
            for sid in members
            if f"{sid}__{label}" in member_cube.columns
        ]
        if cols:
            ft[label] = member_cube[cols].mean(axis=1, skipna=True)
        else:
            ft[label] = np.nan

    family_tables[family] = ft

    print(f"{family}: members={members}")


In [ ]:

# ============================================================
# 9. Evaluate FAMILY predictions on gold
# ============================================================
def safe_auc(y, p):
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    valid = np.isfinite(y) & np.isfinite(p)
    y, p = y[valid], p[valid]
    if len(y) < 3 or len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, p))

def safe_ap(y, p):
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    valid = np.isfinite(y) & np.isfinite(p)
    y, p = y[valid], p[valid]
    if len(y) < 3 or np.sum(y == 1) == 0:
        return np.nan
    return float(average_precision_score(y, p))

def bootstrap_ci(y, p, n_iter=BOOTSTRAP_ITERS):
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    valid = np.isfinite(y) & np.isfinite(p)
    y, p = y[valid], p[valid]

    if len(y) < 3 or len(np.unique(y)) < 2:
        return np.nan, np.nan

    rng = np.random.default_rng(SEED)
    vals = []
    for _ in range(n_iter):
        idx = rng.integers(0, len(y), len(y))
        yy, pp = y[idx], p[idx]
        if len(np.unique(yy)) < 2:
            continue
        vals.append(roc_auc_score(yy, pp))

    if len(vals) < 20:
        return np.nan, np.nan

    return float(np.percentile(vals,2.5)), float(np.percentile(vals,97.5))

family_metric_rows = []

for family, fdf in family_tables.items():
    m = gold[["StudyInstanceUID"] + LABELS].merge(
        fdf, on="StudyInstanceUID", how="left",
        suffixes=("_gold","_pred")
    )

    for label in LABELS:
        y = pd.to_numeric(m[label+"_gold"], errors="coerce").to_numpy(float)
        p = pd.to_numeric(m[label+"_pred"], errors="coerce").to_numpy(float)
        valid = np.isfinite(y) & np.isfinite(p)
        yy, pp = y[valid], p[valid]

        if len(yy)==0:
            continue

        lo, hi = bootstrap_ci(yy, pp)

        family_metric_rows.append({
            "family": family,
            "label": label,
            "n": len(yy),
            "n_positive": int(np.sum(yy==1)),
            "auc": safe_auc(yy,pp),
            "auc_ci_low": lo,
            "auc_ci_high": hi,
            "ap": safe_ap(yy,pp)
        })

family_metrics = pd.DataFrame(family_metric_rows)

family_macro = (
    family_metrics.groupby("family")
    .agg(macro_auc=("auc","mean"), macro_ap=("ap","mean"))
    .sort_values("macro_auc", ascending=False)
    .reset_index()
)

print("Family macro:")
display(family_macro)

print("\nPer-label family AUC:")
display(
    family_metrics.pivot(index="label", columns="family", values="auc")
    .reindex(LABELS).round(3)
)


In [ ]:

# ============================================================
# 10. Build candidate silver-label strategies
# ============================================================
family_cube = all_uids.copy()

for family, fdf in family_tables.items():
    rename = {label: f"{family}__{label}" for label in LABELS}
    family_cube = family_cube.merge(
        fdf.rename(columns=rename),
        on="StudyInstanceUID",
        how="left"
    )

# Identify hybrid family for L0
hybrid_candidates = [f for f in FAMILIES if f == "HYBRID_BLEND"]
HYBRID_FAMILY = hybrid_candidates[0] if hybrid_candidates else FAMILIES[0]

L0 = all_uids.copy()
L1 = all_uids.copy()
L2 = all_uids.copy()
AGREEMENT = all_uids.copy()

for label in LABELS:
    fam_cols = [
        f"{fam}__{label}"
        for fam in FAMILIES
        if f"{fam}__{label}" in family_cube.columns
    ]

    X = family_cube[fam_cols].to_numpy(float)

    # L0: strongest simple reference family
    hcol = f"{HYBRID_FAMILY}__{label}"
    L0[label] = family_cube[hcol] if hcol in family_cube.columns else np.nanmean(X,axis=1)

    # L1: equal weight per independent family
    L1[label] = np.nanmean(X, axis=1)

    # L2: robust family median
    L2[label] = np.nanmedian(X, axis=1)

    # Agreement weight for later loss weighting:
    # 1.0 = families agree closely, lower = disagreement.
    row_mean = np.nanmean(X, axis=1)
    mad = np.nanmean(np.abs(X - row_mean[:,None]), axis=1)
    agreement = np.clip(1.0 - 2.0*mad, 0.0, 1.0)
    AGREEMENT[label] = agreement

print("L0 family:", HYBRID_FAMILY)
print("L1: equal family mean")
print("L2: family median")


In [ ]:

# ============================================================
# 11. EXPERIMENTAL gold-guided blend
#
# This is intentionally NOT the default baseline.
# Weights use lower bootstrap CI above chance, therefore this
# strategy has meta-overfit risk if we later evaluate on the
# same 58 gold studies.
# ============================================================
L3 = all_uids.copy()
gold_guided_weights = []

for label in LABELS:
    fam_scores = []
    fam_names = []

    for family in FAMILIES:
        r = family_metrics[
            (family_metrics["family"]==family) &
            (family_metrics["label"]==label)
        ]

        if len(r)==0:
            continue

        rr = r.iloc[0]
        lower = rr["auc_ci_low"]

        # Conservative reliability:
        # only evidence above chance from lower CI contributes strongly.
        if np.isfinite(lower):
            w = max(float(lower) - 0.5, 0.02)
        else:
            w = 0.02

        fam_names.append(family)
        fam_scores.append(w)

    W = np.asarray(fam_scores,float)
    if W.sum() > 0:
        W = W/W.sum()

    cols = [f"{fam}__{label}" for fam in fam_names]
    X = family_cube[cols].to_numpy(float)

    preds = []
    for row in X:
        valid = np.isfinite(row)
        if not valid.any():
            preds.append(np.nan)
        else:
            ww = W[valid]
            xx = row[valid]
            ww = ww/ww.sum()
            preds.append(float(np.sum(ww*xx)))

    L3[label] = preds

    for fam,w in zip(fam_names,W):
        gold_guided_weights.append({
            "label": label,
            "family": fam,
            "weight": float(w)
        })

gold_guided_weights = pd.DataFrame(gold_guided_weights)
display(gold_guided_weights.head(30))


In [ ]:

# ============================================================
# 12. Candidate evaluation on gold
#
# L0/L1/L2 do NOT use gold to construct scores.
# L3 DOES use gold-derived weights -> evaluation is optimistic.
# ============================================================
candidate_tables = {
    "L0_HYBRID_FAMILY": L0,
    "L1_FAMILY_MEAN": L1,
    "L2_FAMILY_MEDIAN": L2,
    "L3_GOLD_GUIDED_EXPERIMENTAL": L3,
}

candidate_rows = []

for name, cdf in candidate_tables.items():
    m = gold[["StudyInstanceUID"] + LABELS].merge(
        cdf, on="StudyInstanceUID", how="left",
        suffixes=("_gold","_pred")
    )

    for label in LABELS:
        y = m[label+"_gold"].to_numpy(float)
        p = m[label+"_pred"].to_numpy(float)
        candidate_rows.append({
            "candidate": name,
            "label": label,
            "auc": safe_auc(y,p),
            "ap": safe_ap(y,p),
            "uses_gold_for_construction": name.startswith("L3_")
        })

candidate_metrics = pd.DataFrame(candidate_rows)

candidate_macro = (
    candidate_metrics.groupby(
        ["candidate","uses_gold_for_construction"]
    )
    .agg(macro_auc=("auc","mean"), macro_ap=("ap","mean"))
    .reset_index()
    .sort_values("macro_auc", ascending=False)
)

display(candidate_macro)

print("\nPer-label candidate AUC:")
display(
    candidate_metrics.pivot(
        index="label", columns="candidate", values="auc"
    ).reindex(LABELS).round(3)
)


In [ ]:

# ============================================================
# 13. High-disagreement cases for report/domain audit
# ============================================================
case_rows = []

for label in LABELS:
    cols = [f"{fam}__{label}" for fam in FAMILIES]
    X = family_cube[cols].to_numpy(float)

    spread = np.nanmax(X,axis=1) - np.nanmin(X,axis=1)
    std = np.nanstd(X,axis=1)

    tmp = pd.DataFrame({
        "StudyInstanceUID": family_cube["StudyInstanceUID"],
        "label": label,
        "family_score_range": spread,
        "family_score_std": std
    })

    case_rows.append(tmp)

disagreement_cases = pd.concat(case_rows, ignore_index=True)

disagreement_cases = disagreement_cases.merge(
    train[["StudyInstanceUID","Report"]],
    on="StudyInstanceUID",
    how="left"
)

disagreement_cases = disagreement_cases.sort_values(
    ["family_score_range","family_score_std"],
    ascending=False
)

print("Highest disagreement examples:")
display(disagreement_cases.head(100))


In [ ]:

# ============================================================
# 14. Remove 58 gold studies from TRAINING silver exports
# ============================================================
def weak_only(df):
    return df[
        ~df["StudyInstanceUID"].isin(GOLD_UIDS)
    ].reset_index(drop=True)

L0_weak = weak_only(L0)
L1_weak = weak_only(L1)
L2_weak = weak_only(L2)
L3_weak = weak_only(L3)
AGREEMENT_weak = weak_only(AGREEMENT)

assert len(L1_weak) == len(weak)

print("Silver training rows:", len(L1_weak))
print("Gold excluded:", len(gold))



## 15. Cách dùng sau notebook này

### Baseline MRI V1 đề xuất
Dùng:

- target: `L1_family_mean.csv`
- soft labels giữ nguyên `[0,1]`
- **không** threshold thành hard 0/1
- 58 gold studies giữ ngoài optimizer để làm image-level audit

### Controlled label experiments sau đó
Giữ nguyên:
- MRI preprocessing
- backbone
- folds
- seeds
- optimizer

Chỉ thay supervision:

- **Label-E0:** `L0_hybrid_family`
- **Label-E1:** `L1_family_mean`
- **Label-E2:** `L2_family_median`
- **Label-E3:** `L1_family_mean` + `agreement_weight` trong loss
- **Label-E4:** `L3_gold_guided_EXPERIMENTAL` (chỉ sau cùng, vì meta-overfit risk)

Nếu `Label-E1/E2/E3` thắng L0 ổn định qua nhiều seeds, mới giữ consensus cho pipeline chính.


In [ ]:

# ============================================================
# 16. Export
# ============================================================
OUT = Path("/kaggle/working/rsna_silver_label_v1")
OUT.mkdir(parents=True, exist_ok=True)

source_manifest.to_csv(OUT/"source_family_manifest.csv", index=False)
pairwise_dependency.to_csv(OUT/"source_dependency_audit.csv", index=False)
family_metrics.to_csv(OUT/"family_gold_metrics.csv", index=False)
family_macro.to_csv(OUT/"family_gold_macro.csv", index=False)
candidate_metrics.to_csv(OUT/"candidate_gold_metrics.csv", index=False)
candidate_macro.to_csv(OUT/"candidate_gold_macro.csv", index=False)
gold_guided_weights.to_csv(OUT/"gold_guided_weights_EXPERIMENTAL.csv", index=False)

L0_weak.to_csv(OUT/"L0_hybrid_family.csv", index=False)
L1_weak.to_csv(OUT/"L1_family_mean_RECOMMENDED.csv", index=False)
L2_weak.to_csv(OUT/"L2_family_median.csv", index=False)
L3_weak.to_csv(OUT/"L3_gold_guided_EXPERIMENTAL.csv", index=False)
AGREEMENT_weak.to_csv(OUT/"L1_agreement_weights.csv", index=False)

disagreement_cases.head(2000).to_csv(
    OUT/"high_disagreement_cases.csv", index=False
)

summary = []
summary.append("RSNA KNEE — SILVER LABEL CONSTRUCTION v1")
summary.append("="*72)
summary.append(f"Train studies: {len(train)}")
summary.append(f"Gold studies excluded from silver training: {len(gold)}")
summary.append(f"Weak silver-training studies: {len(L1_weak)}")
summary.append("")
summary.append("LEAKY/REFERENCE SOURCES EXCLUDED")
summary.append(str(sorted(LEAKY_SOURCES)))
summary.append("")
summary.append("INDEPENDENT FAMILIES")
for fam in FAMILIES:
    members = source_manifest.loc[
        source_manifest["family"].eq(fam),
        "source_id"
    ].tolist()
    summary.append(f"- {fam}: {members}")
summary.append("")
summary.append("FAMILY GOLD MACRO")
for _,r in family_macro.iterrows():
    summary.append(
        f"- {r['family']}: macro_auc={r['macro_auc']:.4f}, "
        f"macro_ap={r['macro_ap']:.4f}"
    )
summary.append("")
summary.append("CANDIDATE GOLD MACRO")
for _,r in candidate_macro.iterrows():
    flag = " [USES GOLD -> OPTIMISTIC]" if r["uses_gold_for_construction"] else ""
    summary.append(
        f"- {r['candidate']}: macro_auc={r['macro_auc']:.4f}, "
        f"macro_ap={r['macro_ap']:.4f}{flag}"
    )
summary.append("")
summary.append("RECOMMENDED FOR BASELINE MRI V1:")
summary.append("- L1_family_mean_RECOMMENDED.csv")
summary.append("- keep labels soft")
summary.append("- keep 58 gold studies outside optimizer")
summary.append("- optional later experiment: L1_agreement_weights.csv")

text = "\n".join(summary)
(OUT/"RESULTS_TO_SEND_BACK.txt").write_text(text, encoding="utf-8")
print(text)

ZIP_PATH = Path("/kaggle/working/RSNA_Knee_Silver_Label_v1_Artifacts.zip")
with zipfile.ZipFile(ZIP_PATH,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.relative_to(OUT))

print("\nZIP:", ZIP_PATH)
print("Send this ZIP back to ChatGPT.")
